# Cross-dataset concept transfer

Apply saved source concepts to matched-gene target datasets without retraining. Evaluate six transfer directions and robustness to changes in one target cell-type abundance at a time. Run the sections in order.
## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.


In [ ]:
from pathlib import Path
import json
import hashlib
import platform
import importlib.metadata
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from IPython.display import display

# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
DATA_DIR = input_path("transfer.inputs", "transfer")
OUTPUT_DIR = TUTORIAL_DIR / "outputs/transfer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_K = 100
MIN_CELLS = 100
FRACTIONS = [1.0, 0.5, 0.2, 0.1]
SEEDS = [1, 2, 3, 4, 5]
RUN_SHIFTS = True
RUN_PLOTS = True
PAIRS = [("Lung", "Reyfman", "Vieira"), ("Pancreas", "Muraro", "Wang"), ("Brain", "Romanov", "Zeisel")]
DATASET_IDS = {"Reyfman": "Reyfman_Misharin_2018", "Vieira": "Vieira_Teichmann_2019", "Muraro": "Muraro", "Wang": "wang", "Romanov": "Romanov", "Zeisel": "zeisel"}


## 1. Load matched inputs and score concepts

Use the first 100 listed genes of each source concept, matching gene symbols to target columns. Standardize genes within the evaluated target cohort using population standard deviation plus 1e-6, then average matched gene scores. Recompute standardization after each subsampling. Keep all target cell types, including types absent from the source.

Report ARI, NMI, AMI, and purity separately. The original evaluation function labeled AMI as NMI; `NMI_reported` in the archived comparison table preserves the supplied values. `MIN_CELLS=100` follows the manuscript; set it to 150 to use the supplied notebook's threshold.
Cell-barcode labels are aligned by ID. Muraro, Wang, Romanov, and Zeisel use sequential numeric label indices; these follow expression row order, as in the supplied analysis, with equal row counts required.


In [ ]:
def load_concepts(name):
    path = CONCEPT_DIR / f"GPT5_{name}_concepts_match.json"
    concepts = load_saved_concepts(path)
    return [dict(c, genes=c["genes"][:TOP_K]) for c in concepts]

def load_dataset(name, required_genes):
    dataset = DATASET_IDS[name]
    path = DATA_DIR / f"{dataset}_HIGHPRE_10000_match.csv"
    header = pd.read_csv(path, nrows=0).columns.tolist()
    if len(header[1:]) != len(set(header[1:])):
        raise ValueError(f"Duplicate gene columns: {path}")
    genes = [g for g in header[1:] if g in required_genes]
    frame = pd.read_csv(path, usecols=[header[0], *genes], index_col=0)
    labels = pd.read_csv(DATA_DIR / f"{dataset}_cell_anno.csv", index_col=0)
    if not frame.index.is_unique or not labels.index.is_unique:
        raise ValueError(f"Duplicate cell identifiers: {name}")
    if not frame.index.equals(labels.index):
        if set(frame.index) == set(labels.index):
            labels = labels.loc[frame.index]
        elif len(labels) == len(frame) and (np.array_equal(labels.index.to_numpy(), np.arange(len(labels))) or np.array_equal(labels.index.to_numpy(), np.arange(1, len(labels) + 1))):
            labels.index = frame.index
        else:
            raise ValueError(f"Expression and label cell identifiers differ: {name}")
    if labels.iloc[:, 0].isna().any():
        raise ValueError(f"Missing labels: {name}")
    X = frame.to_numpy(dtype=float)
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite expression: {name}")
    return X, labels.iloc[:, 0].astype(str).to_numpy(), frame.columns.tolist(), frame.index.to_numpy(dtype=str), header[1:]

def score_concepts(concepts, X, genes):
    index = {g: i for i, g in enumerate(genes)}
    z = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)
    scores = np.zeros((len(X), len(concepts)))
    for i, concept in enumerate(concepts):
        selected = [index[g] for g in concept["genes"] if g in index]
        if selected:
            scores[:, i] = z[:, selected].mean(axis=1)
    return scores

def evaluate(y, scores):
    prediction = scores.argmax(axis=1)
    counts = pd.crosstab(pd.Series(y), pd.Series(prediction))
    return dict(Purity=float(counts.max(axis=0).sum() / len(y)),
                ARI=adjusted_rand_score(y, prediction),
                NMI=normalized_mutual_info_score(y, prediction),
                AMI=adjusted_mutual_info_score(y, prediction))

def subsample_indices(y, cell_type, fraction, seed):
    rng = np.random.default_rng(seed)
    selected = np.flatnonzero(y == cell_type)
    other = np.flatnonzero(y != cell_type)
    if fraction != 1.0:
        selected = rng.choice(selected, size=max(1, int(len(selected) * fraction)), replace=False)
    indices = np.concatenate([other, selected])
    rng.shuffle(indices)
    return indices


## 2. Evaluate each pair in both directions

For each target cell type with at least `MIN_CELLS` cells, retain 50%, 20%, or 10% of that type while keeping all other cells. Repeat with five seeds. Save one full-target baseline per direction and record retained row indices and the target cell-ID array for every subsample. Only genes used by either concept set are loaded; gene-wise scoring is unchanged.

In [ ]:
transfer_rows = []
within_rows = []
shift_rows = []
input_rows = []
for tissue, left, right in PAIRS:
    concept_sets = {name: load_concepts(name) for name in [left, right]}
    required = {g for concepts in concept_sets.values() for c in concepts for g in c["genes"]}
    loaded = {name: load_dataset(name, required) for name in [left, right]}
    if set(loaded[left][4]) != set(loaded[right][4]):
        raise ValueError(f"Matched gene universes differ: {left}, {right}")
    for name in [left, right]:
        X, y, genes, ids, all_genes = loaded[name]
        input_rows.append(dict(dataset=name, original_dataset=DATASET_IDS[name], n_cells=len(y), n_genes=len(all_genes)))
        counts = pd.Series(y).value_counts().rename_axis("cell_type").reset_index(name="n_cells")
        counts.to_csv(OUTPUT_DIR / f"{name}_label_counts.csv", index=False)
        within_rows.append(dict(dataset=name, **evaluate(y, score_concepts(concept_sets[name], X, genes))))
    for source, target in [(left, right), (right, left)]:
        X, y, genes, ids, _ = loaded[target]
        concepts = concept_sets[source]
        out = OUTPUT_DIR / f"{source}_to_{target}"
        out.mkdir(exist_ok=True)
        scores = score_concepts(concepts, X, genes)
        result = dict(source=source, target=target, tissue=tissue, n_cells=len(y), **evaluate(y, scores))
        transfer_rows.append(result)
        pd.DataFrame(dict(cell_id=ids, cell_type=y, concept=np.asarray([c["name"] for c in concepts])[scores.argmax(axis=1)])).to_csv(out / "assignments.csv", index=False)
        np.savez_compressed(out / "concept_scores.npz", scores=scores, names=np.asarray([c["name"] for c in concepts]))
        matches = [dict(concept=c["name"], requested=c["genes"], matched=[g for g in c["genes"] if g in genes]) for c in concepts]
        (out / "gene_matches.json").write_text(json.dumps(matches, indent=2))
        shift_rows.append(dict(result, shifted_type="baseline", fraction=1.0, seed=1, original_n=len(y), remaining_n=len(y)))
        if RUN_SHIFTS:
            counts = pd.Series(y).value_counts()
            retained = {}
            for cell_type in counts[counts >= MIN_CELLS].index:
                for fraction in FRACTIONS[1:]:
                    for seed in SEEDS:
                        indices = subsample_indices(y, cell_type, fraction, seed)
                        key = f"sample_{len(retained):04d}"
                        retained[key] = indices
                        metrics = evaluate(y[indices], score_concepts(concepts, X[indices], genes))
                        shift_rows.append(dict(source=source, target=target, tissue=tissue, shifted_type=cell_type,
                            fraction=fraction, seed=seed, original_n=int(counts[cell_type]),
                            remaining_n=int(np.sum(y[indices] == cell_type)), n_cells=len(indices), sample_key=key, **metrics))
            np.savez_compressed(out / "subsample_indices.npz", cell_ids=ids, **retained)
        print(source, "->", target, result)
    pd.DataFrame(transfer_rows).to_csv(OUTPUT_DIR / "transfer_metrics.csv", index=False)
    pd.DataFrame(shift_rows).to_csv(OUTPUT_DIR / "label_shift_runs.csv", index=False)
    del loaded
    gc.collect()
transfer_results = pd.DataFrame(transfer_rows)
within_results = pd.DataFrame(within_rows)
shift_results = pd.DataFrame(shift_rows)
transfer_results.to_csv(OUTPUT_DIR / "transfer_metrics.csv", index=False)
within_results.to_csv(OUTPUT_DIR / "within_dataset_metrics.csv", index=False)
shift_results.to_csv(OUTPUT_DIR / "label_shift_runs.csv", index=False)
pd.DataFrame(input_rows).to_csv(OUTPUT_DIR / "dataset_inputs.csv", index=False)
display(transfer_results)


## 3. Summarize composition shifts

Summarize mean and sample standard deviation across cell-type/seed combinations for each direction and retention fraction. Also report each shifted cell type separately. The full-target baseline is evaluated once.

In [ ]:
metric_columns = ["Purity", "ARI", "NMI", "AMI"]
summary = shift_results.groupby(["source", "target", "fraction"])[metric_columns].agg(["mean", "std"])
detail = shift_results[shift_results.shifted_type != "baseline"].groupby(["source", "target", "shifted_type", "fraction"])[metric_columns].agg(["mean", "std"])
summary.to_csv(OUTPUT_DIR / "label_shift_summary.csv")
detail.to_csv(OUTPUT_DIR / "label_shift_by_cell_type.csv")
display(summary)
display(detail)


## 4. Plot transfer robustness

Plot recomputed ARI and NMI using the original colors, line styles, markers, axes, and error bands. Error bands show standard deviations across shifted cell types and seeds.

In [ ]:
if RUN_PLOTS and RUN_SHIFTS:
    fractions = np.array(FRACTIONS)
    xlabels = ["100", "50", "20", "10"]
    data = {}
    for tissue, left, right in [("Brain", "Romanov", "Zeisel"), ("Pancreas", "Muraro", "Wang"), ("Lung", "Reyfman", "Vieira")]:
        for direction, (source, target) in enumerate([(left, right), (right, left)]):
            d = {"pair": tissue, "direction": direction}
            for metric in ["ARI", "NMI"]:
                d[metric] = [summary.loc[(source, target, fraction), (metric, "mean")] for fraction in FRACTIONS]
                d[f"{metric}_std"] = [0.0 if fraction == 1.0 else summary.loc[(source, target, fraction), (metric, "std")] for fraction in FRACTIONS]
            data[f"{source} → {target}"] = d
    
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7.5,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })
    
    colors = {
        "Brain": "#4C78A8",
        "Pancreas": "#E45756",
        "Lung": "#59A14F",
    }
    
    linestyles = ["-", "--"]
    markers = ["o", "s"]
    
    
    
    fig, axes = plt.subplots(
        1, 2,
        figsize=(7.2, 3.0),
        sharex=True
    )
    
    metrics = ["ARI", "NMI"]
    
    for ax, metric in zip(axes, metrics):
    
        for name, d in data.items():
    
            y = np.asarray(d[metric])
            yerr = np.asarray(d[f"{metric}_std"])
    
            color = colors[d["pair"]]
            ls = linestyles[d["direction"]]
            marker = markers[d["direction"]]
    
            ax.plot(
                fractions,
                y,
                color=color,
                linestyle=ls,
                marker=marker,
                markersize=4.5,
                linewidth=1.5,
                markeredgewidth=0.7,
                markeredgecolor="white",
                label=name
            )
    
            ax.fill_between(
                fractions,
                y - yerr,
                y + yerr,
                color=color,
                alpha=0.10,
                linewidth=0
            )
    
        ax.set_xlim(1.04, 0.06)
    
        ax.set_xticks(fractions)
        ax.set_xticklabels(xlabels)
    
        ax.set_xlabel(
            "Retained fraction of perturbed population (%)"
        )
    
        ax.set_ylabel(metric)
    
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
        ax.tick_params(
            direction="out",
            length=3,
            pad=3
        )
    
        ax.grid(
            axis="y",
            linewidth=0.45,
            alpha=0.20
        )
    
    
    
    
    
    
    
    handles, labels = axes[1].get_legend_handles_labels()
    
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        frameon=False,
        handlelength=2.8,
        columnspacing=1.5
    )
    
    
    
    plt.subplots_adjust(
        left=0.10,
        right=0.98,
        top=0.95,
        bottom=0.28,
        wspace=0.30
    )
    
    plt.savefig(
        OUTPUT_DIR / "label_shift_transfer_ARI_NMI.pdf",
        bbox_inches="tight"
    )
    
    
    plt.show()


## 5. Plot the archived method comparison

The supplied comparison table contains reported results for scConcept, scVI, scVI-LD, and scETM. Plot these archived values separately from recomputed metrics. The single-panel overall score is the mean of the two reported metrics. Baseline training and inference are not implemented in this notebook; the reported NMI column retains its original metric label.

In [ ]:
if RUN_PLOTS:
    df = pd.read_csv(TUTORIAL_DIR / "Inputs/transfer/reported_transfer_results.csv").rename(columns={"NMI_reported": "NMI"})
    plt.rcParams.update({
        "font.size": 11,
        "font.family": "Arial",
        "axes.linewidth": 1.2,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,
        "pdf.fonttype": 42,
    })
    
    colors = {
        "scConcept": "#AB4843",
        "scVI": "#CE8540",
        "scVI-LD": "#E2BC91",
        "scETM": "#E3A8A5",
    }
    
    methods = ["scConcept", "scVI", "scVI-LD", "scETM"]
    
    
    def plot_metric(ax, metric, title):
    
        groups = df["dataset"].unique()
        x = np.arange(len(groups))
        width = 0.18
    
        for i, m in enumerate(methods):
    
            vals = [
                df[(df.dataset==g)&(df.method==m)][metric].values[0]
                for g in groups
            ]
    
            ax.bar(
                x + (i-1.5)*width,
                vals,
                width,
                color=colors[m],
                edgecolor="black",
                linewidth=0.6,
                label=m
            )
    
        ax.set_xticks(x)
        ax.set_xticklabels(groups, rotation=30, ha="right")
        ax.set_ylabel(metric)
        ax.set_ylim(0,1)
    
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
        ax.set_title(title, pad=8)
    
    
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    
    plot_metric(axes[0], "ARI", "Cross-dataset ARI")
    plot_metric(axes[1], "NMI", "Cross-dataset NMI")
    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        ncol=4,
        loc="upper center",
        frameon=False,
        bbox_to_anchor=(0.5,1.08)
    )
    
    plt.tight_layout(rect=[0,0,1,0.94])
    
    plt.savefig(OUTPUT_DIR / "cross_dataset_reported_comparison.pdf", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
if RUN_PLOTS:
    df = pd.read_csv(TUTORIAL_DIR / "Inputs/transfer/reported_transfer_results.csv").rename(columns={"NMI_reported": "NMI"})
    df["method"] = df["method"].replace({"scVI-LD": "scVILD"})
    df["Score"] = (df["ARI"] + df["NMI"]) / 2
    
    
    plt.rcParams.update({
        "font.size": 11,
        "font.family": "Arial",
        "axes.linewidth": 1.2,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,
        "pdf.fonttype": 42,  # illustrator 可编辑
    })
    
    colors = {
        "scConcept": "#AB4843",
        "scVI":      "#CE8540",
        "scVILD":    "#E2BC91",
        "scETM":     "#E3A8A5",
    }
    
    methods = ["scConcept", "scVI", "scVILD", "scETM"]
    
    
    groups = df["dataset"].unique()
    x = np.arange(len(groups))
    width = 0.18
    
    fig, ax = plt.subplots(figsize=(12, 4.2))
    
    for i, m in enumerate(methods):
        vals = [
            df[(df.dataset == g) & (df.method == m)]["Score"].values[0]
            for g in groups
        ]
        ax.bar(
            x + (i - 1.5) * width,
            vals,
            width=width,
            color=colors[m],
            edgecolor="black",
            linewidth=0.6,
            label=m,
        )
    
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=30, ha="right")
    ax.set_ylabel("Overall")
    ax.set_ylim(0, 1)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        ncol=4,
        loc="upper center",
        frameon=False,
        bbox_to_anchor=(0.5, 1.08)
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(OUTPUT_DIR / "cross_protocol_reported_score.pdf", dpi=300, bbox_inches="tight")
    plt.show()


## Output guide

Paths below are relative to `outputs/transfer`.

| Analysis | Main output |
|---|---|
| Six transfer directions | `transfer_metrics.csv` |
| Within-dataset reference performance | `within_dataset_metrics.csv` |
| Cell assignments and concept scores | `<source>_to_<target>/assignments.csv`, `<source>_to_<target>/concept_scores.npz` |
| Individual subsampling runs | `label_shift_runs.csv` |
| Overall and cell-type summaries | `label_shift_summary.csv`, `label_shift_by_cell_type.csv` |
| Retained indices and target cell IDs | `<source>_to_<target>/subsample_indices.npz` |
| Composition-shift performance plot | `label_shift_transfer_ARI_NMI.pdf` |
| Archived method comparisons | `cross_dataset_reported_comparison.pdf`, `cross_protocol_reported_score.pdf` |
| Input sizes and target label distributions | `dataset_inputs.csv`, `<dataset>_label_counts.csv` |

The final cell saves the run configuration and package versions.

## 6. Save run settings

In [ ]:
configuration = dict(python=platform.python_version(),
    packages={p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scikit-learn", "matplotlib"]},
    data_dir=str(DATA_DIR), top_k=TOP_K, min_cells=MIN_CELLS, fractions=FRACTIONS, seeds=SEEDS,
    pairs=PAIRS, run_shifts=RUN_SHIFTS,
    concept_sha256={name: hashlib.sha256((CONCEPT_DIR / f"GPT5_{name}_concepts_match.json").read_bytes()).hexdigest() for name in DATASET_IDS})
configuration["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(configuration, indent=2))
print("Outputs:", OUTPUT_DIR)
